# Supervised KAN Collision-Risk Prediction
## Kolmogorov–Arnold Network (PyKAN) · ESA Collision Avoidance Challenge Dataset

This notebook trains a **supervised Kolmogorov–Arnold Network (KAN) regression model** using the official **ESA Space Debris Office Collision Avoidance Challenge (Kelvins)** dataset (`train_data.csv`).

### Scientific & Mission Context
Between 2015 and 2019, ESA's Space Debris Office monitored operational spacecraft (including *Aeolus*, *CryoSat-2*, and the *Swarm* constellation) in Low Earth Orbit (LEO). Whenever a potential close approach was detected by the US Space Surveillance Network, a sequence of **Conjunction Data Messages (CDMs)** was generated as the encounter evolved over time.

Each CDM provides 103 recorded characteristics: relative encounter geometry, relative velocity, 6-DoF state covariances for target and chaser objects, solar and atmospheric activity indices, and an analytical collision risk metric.

### Key Implementation Principles
1. **Target Semantics:** The target column `risk` represents $\log_{10}(P_c)$, the self-computed base-10 logarithm of collision probability at each CDM epoch. It is preserved in its documented scale (no duplicate log transform).
2. **Strict Leakage Prevention:** Data splitting is grouped strictly by `event_id` so all repeated CDMs of an encounter remain in one fold. Columns that leak target proxies or future states (`max_risk_estimate`, `max_risk_scaling`) are rigorously excluded from features.
3. **Model Framework:** Uses **PyKAN** (`MultKAN`) with learnable B-spline activation functions on each network edge, compared against a Ridge regression baseline.
4. **Dataset Separation:** The private test set (`test_data_private.csv`) and public test set are kept completely separate and never accessed during training, feature selection, or tuning.

## Cell 0 — Environment & Kernel Verification

In [44]:
import sys, os
print('Python Executable :', sys.executable)
print('Python Version    :', sys.version.split()[0])

# Ensure running inside project .venv
assert '.venv' in sys.executable or 'venv' in sys.executable, \
    'WARNING: Not running inside the project .venv! Please select the "ICPS .venv" kernel.'
print('✓ Active kernel is verified inside the project virtual environment (.venv)')

Python Executable : c:\Users\sreek\Sreekar\folder\.venv\Scripts\python.exe
Python Version    : 3.12.6
✓ Active kernel is verified inside the project virtual environment (.venv)


## Cell 1 — Project Paths & Directory Setup

In [45]:
import os, sys

# Resolve paths relative to the project root
NOTEBOOK_DIR = os.path.abspath(os.path.dirname('__file__') if '__file__' in dir() else '.')
PROJECT_ROOT = os.path.abspath(os.path.join(NOTEBOOK_DIR, '..'))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

DATA_RAW_CSV      = os.path.join(PROJECT_ROOT, 'data', 'raw', 'esa_collision_avoidance', 'kelvins_competition_data', 'train_data', 'train_data.csv')
DATA_PROCESSED_DIR = os.path.join(PROJECT_ROOT, 'data', 'processed')
MODELS_DIR         = os.path.join(NOTEBOOK_DIR, 'saved_models')
PLOTS_DIR          = os.path.join(NOTEBOOK_DIR, 'plots')

os.makedirs(DATA_PROCESSED_DIR, exist_ok=True)
os.makedirs(MODELS_DIR, exist_ok=True)
os.makedirs(PLOTS_DIR, exist_ok=True)

print(f'Project root       : {PROJECT_ROOT}')
print(f'Raw training CSV   : {DATA_RAW_CSV}')
print(f'Processed data dir : {DATA_PROCESSED_DIR}')
print(f'Models saved dir   : {MODELS_DIR}')
print(f'Plots saved dir    : {PLOTS_DIR}')
assert os.path.exists(DATA_RAW_CSV), f'Raw dataset not found at {DATA_RAW_CSV}'
print('✓ All project directories confirmed.')

Project root       : c:\Users\sreek\Sreekar\folder\satellite-collision-avoidance-system
Raw training CSV   : c:\Users\sreek\Sreekar\folder\satellite-collision-avoidance-system\data\raw\esa_collision_avoidance\kelvins_competition_data\train_data\train_data.csv
Processed data dir : c:\Users\sreek\Sreekar\folder\satellite-collision-avoidance-system\data\processed
Models saved dir   : c:\Users\sreek\Sreekar\folder\satellite-collision-avoidance-system\notebooks\saved_models
Plots saved dir    : c:\Users\sreek\Sreekar\folder\satellite-collision-avoidance-system\notebooks\plots
✓ All project directories confirmed.


## Cell 2 — Dataset Inspection & Summary Statistics

We load `train_data.csv` and systematically audit row/column counts, memory footprint, data types, missing value percentages, and duplicate records.

In [46]:
import pandas as pd
import numpy as np

print('Loading ESA collision avoidance dataset...')
df_raw = pd.read_csv(DATA_RAW_CSV)
n_rows, n_cols = df_raw.shape

print(f'=== Dataset Dimensions ===')
print(f'Total rows (CDMs) : {n_rows:,}')
print(f'Total columns     : {n_cols}')
print(f'Duplicate rows    : {df_raw.duplicated().sum()}')
print(f'Memory footprint  : {df_raw.memory_usage(deep=True).sum() / (1024**2):.2f} MB')

print(f'\n=== Data Types Summary ===')
print(df_raw.dtypes.value_counts().to_string())

# Check columns with missing values
null_series = df_raw.isnull().sum()
null_cols = null_series[null_series > 0].sort_values(ascending=False)
print(f'\n=== Missing Values Audit ===')
print(f'Columns with missing values: {len(null_cols)} / {n_cols}')
print('Top 10 columns by missing percentage:')
for col, cnt in null_cols.head(10).items():
    pct = (cnt / n_rows) * 100
    print(f'  {col:30s} : {cnt:6d} nulls ({pct:5.2f}%)')

# Verify key critical columns have zero nulls
critical_cols = ['event_id', 'time_to_tca', 'risk', 'miss_distance', 'relative_speed']
for c in critical_cols:
    assert df_raw[c].isnull().sum() == 0, f'Critical column {c} has nulls!'
print('✓ Critical orbital columns (event_id, time_to_tca, risk, miss_distance) are 100% complete.')

Loading ESA collision avoidance dataset...
=== Dataset Dimensions ===
Total rows (CDMs) : 162,634
Total columns     : 103
Duplicate rows    : 0
Memory footprint  : 135.16 MB

=== Data Types Summary ===
float64    98
int64       4
str         1

=== Missing Values Audit ===
Columns with missing values: 50 / 103
Top 10 columns by missing percentage:
  c_rcs_estimate                 :  52841 nulls (32.49%)
  c_crdot_r                      :   9241 nulls ( 5.68%)
  c_cndot_n                      :   9241 nulls ( 5.68%)
  c_cndot_rdot                   :   9241 nulls ( 5.68%)
  c_cndot_tdot                   :   9241 nulls ( 5.68%)
  c_sigma_rdot                   :   9241 nulls ( 5.68%)
  c_sigma_ndot                   :   9241 nulls ( 5.68%)
  c_sigma_tdot                   :   9241 nulls ( 5.68%)
  c_cndot_t                      :   9241 nulls ( 5.68%)
  c_cndot_r                      :   9241 nulls ( 5.68%)
✓ Critical orbital columns (event_id, time_to_tca, risk, miss_distance) are 100%

## Cell 3 — Target Semantics & Distribution Analysis (`risk`)

### Definition of `risk`:
According to official ESA Space Debris Office dataset documentation (`raw_data_2015-2019.txt`):
> `risk`: *Self-computed value at the epoch of each CDM [base 10 log].*

- `risk` is **already in base-10 logarithmic scale**: $\text{risk} = \log_{10}(P_c)$.
- It is **not** raw probability in $[0, 1]$.
- It is self-computed by the operational ESA CDM software, **not** equivalent to the project's local Foster (1992) numerical solver.
- Values range from $-30.0$ to $-1.44$. The value $-30.0$ is the operational numerical underflow clamp (representing $P_c \le 10^{-30}$ or zero risk).
- Because it is already in $\log_{10}$ space, no additional log transform is needed.

In [47]:
risk_col = df_raw['risk']

print('=== Target (`risk`) Statistical Profile ===')
print(risk_col.describe().round(4).to_string())

n_floor = (risk_col <= -30.0).sum()
pct_floor = (n_floor / len(risk_col)) * 100
print(f'\nNumerical underflow floor (-30.0): {n_floor:,} rows ({pct_floor:.2f}%)')
print(f'Active risk (> -30.0)            : {len(risk_col) - n_floor:,} rows ({100 - pct_floor:.2f}%)')

quantiles = risk_col.quantile([0.01, 0.05, 0.10, 0.25, 0.50, 0.75, 0.90, 0.95, 0.99])
print('\nKey Quantiles of risk:')
for q, val in quantiles.items():
    print(f'  {int(q*100):2d}% percentile: {val:8.4f}')

=== Target (`risk`) Statistical Profile ===
count    162634.0000
mean        -19.3406
std          10.0116
min         -30.0000
25%         -30.0000
50%         -17.8706
75%          -9.1733
max          -1.4429

Numerical underflow floor (-30.0): 67,240 rows (41.34%)
Active risk (> -30.0)            : 95,394 rows (58.66%)

Key Quantiles of risk:
   1% percentile: -30.0000
   5% percentile: -30.0000
  10% percentile: -30.0000
  25% percentile: -30.0000
  50% percentile: -17.8706
  75% percentile:  -9.1733
  90% percentile:  -6.5761
  95% percentile:  -5.7331
  99% percentile:  -4.4808


## Cell 4 — Encounter Event Grouping & Time Series Structure

The dataset consists of multiple unique collision events identified by `event_id`. Each collision event contains a time series of successive CDMs as time-to-TCA decreases.

In [48]:
n_events = df_raw['event_id'].nunique()
rows_per_event = df_raw.groupby('event_id').size()

print(f'Total unique encounter events (event_id) : {n_events:,}')
print(f'CDMs per event — Min                    : {rows_per_event.min()}')
print(f'CDMs per event — Max                    : {rows_per_event.max()}')
print(f'CDMs per event — Median                 : {rows_per_event.median():.1f}')
print(f'CDMs per event — Mean                   : {rows_per_event.mean():.2f}')

# Display sample time series for an example event
sample_event_id = df_raw['event_id'].iloc[0]
sample_ts = df_raw[df_raw['event_id'] == sample_event_id][['event_id', 'time_to_tca', 'miss_distance', 'relative_speed', 'risk']]
print(f'\nSample time series for event_id={sample_event_id} (ordered by CDM arrival):')
print(sample_ts.sort_values('time_to_tca', ascending=False).to_string(index=False))

Total unique encounter events (event_id) : 13,154
CDMs per event — Min                    : 1
CDMs per event — Max                    : 23
CDMs per event — Median                 : 13.0
CDMs per event — Mean                   : 12.36

Sample time series for event_id=0 (ordered by CDM arrival):
 event_id  time_to_tca  miss_distance  relative_speed       risk
        0     1.566798        14923.0         13792.0 -10.204955
        0     1.207494        14544.0         13792.0 -10.355758
        0     0.952193        14475.0         13792.0 -10.345631
        0     0.579669        14579.0         13792.0 -10.337809
        0     0.257806        14510.0         13792.0 -10.391260


## Cell 5 — Target Leakage Investigation & Column Classification

### Column Classification:
1. **Identifiers (Must Exclude from Features):**
   - `event_id`: Identifies the encounter. Used solely for grouped dataset splitting.
   - `mission_id`: Satellite mission ID (categorical identifier).
2. **Target (Exclude from Features):**
   - `risk`: The supervised regression target.
3. **Leakage & Proxy Columns (Must Exclude from Features):**
   - `max_risk_estimate`: Maximum collision probability obtained by scaling the combined covariance. In single-CDM prediction, it acts as an analytical shortcut/proxy for the target; in time-series forecasting, it leaks future encounter state.
   - `max_risk_scaling`: Covariance scaling factor used to compute maximum collision risk.
4. **Categorical Fields:**
   - `c_object_type`: Space debris/object classification (`DEBRIS`, `UNKNOWN`, `PAYLOAD`, `ROCKET BODY`, `TBA`). Encoded via one-hot dummy variables.
5. **Physical Orbital & Uncertainty Features (Preserved):**
   - Relative encounter geometry, RTN position and velocity components, orbit parameters, and covariance standard deviations.

In [49]:
corr_leakage = df_raw[['risk', 'max_risk_estimate', 'max_risk_scaling', 'miss_distance', 'time_to_tca']].corr()
print('Correlation Matrix with Target (risk):')
print(corr_leakage[['risk']].round(4).to_string())

# Check c_object_type categories
print('\nCategorical distribution of c_object_type:')
print(df_raw['c_object_type'].value_counts(dropna=False).to_string())

# Explicit list of excluded columns
EXCLUDED_COLS = ['event_id', 'mission_id', 'risk', 'max_risk_estimate', 'max_risk_scaling']
print(f'\n✓ Explicitly excluded leakage/target/identifier columns: {EXCLUDED_COLS}')

Correlation Matrix with Target (risk):
                     risk
risk               1.0000
max_risk_estimate  0.2238
max_risk_scaling  -0.0630
miss_distance     -0.1583
time_to_tca        0.5181

Categorical distribution of c_object_type:
c_object_type
DEBRIS         94470
UNKNOWN        50542
PAYLOAD        15498
ROCKET BODY     1947
TBA              177

✓ Explicitly excluded leakage/target/identifier columns: ['event_id', 'mission_id', 'risk', 'max_risk_estimate', 'max_risk_scaling']


## Cell 6 — Feature Selection & Leakage-Free Preprocessing Pipeline

We select 24 physically grounded numerical features spanning encounter timing, relative geometry, relative velocity, orbital elements, hard-body dimensions, and covariance uncertainties, plus one-hot encoded `c_object_type` categories.

Preprocessing handles missing values via median imputation and normalizes features using `StandardScaler` fitted **exclusively on the training split**.

In [50]:
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer

NUMERICAL_FEATURES = [
    'time_to_tca', 'miss_distance', 'relative_speed',
    'relative_position_r', 'relative_position_t', 'relative_position_n',
    'relative_velocity_r', 'relative_velocity_t', 'relative_velocity_n',
    'geocentric_latitude', 'azimuth', 'elevation', 'mahalanobis_distance',
    't_j2k_sma', 't_j2k_ecc', 't_j2k_inc',
    'c_j2k_sma', 'c_j2k_ecc', 'c_j2k_inc',
    't_span', 'c_span',
    't_sigma_r', 'c_sigma_r', 't_sigma_t', 'c_sigma_t', 't_sigma_n', 'c_sigma_n',
    't_position_covariance_det', 'c_position_covariance_det',
]
CATEGORICAL_FEATURE = 'c_object_type'
TARGET_COL = 'risk'

print(f'Selected numerical features  : {len(NUMERICAL_FEATURES)}')
print(f'Selected categorical feature : {CATEGORICAL_FEATURE}')
print(f'Target column                : {TARGET_COL}')

Selected numerical features  : 29
Selected categorical feature : c_object_type
Target column                : risk


## Cell 7 — Grouped Train/Validation/Test Split (by `event_id`)

To strictly avoid data leakage across repeated CDMs, data are partitioned by `event_id` into:
- **Train (70% of events)**
- **Validation (15% of events)**
- **Test (15% of events)**

Preprocessed datasets are saved under `data/processed/`.

In [51]:
from sklearn.model_selection import GroupShuffleSplit

np.random.seed(42)

# 1. One-hot encode categorical feature
df_encoded = pd.get_dummies(df_raw, columns=[CATEGORICAL_FEATURE], drop_first=True, dtype=float)
encoded_cat_cols = [c for c in df_encoded.columns if c.startswith(CATEGORICAL_FEATURE + '_')]
ALL_FEATURE_COLS = NUMERICAL_FEATURES + encoded_cat_cols

print(f'Total feature count (numerical + one-hot): {len(ALL_FEATURE_COLS)}')

# 2. First split: Train (70%) vs Temp (30%)
gss_outer = GroupShuffleSplit(n_splits=1, test_size=0.30, random_state=42)
train_idx, temp_idx = next(gss_outer.split(df_encoded, groups=df_encoded['event_id']))

df_train = df_encoded.iloc[train_idx].copy()
df_temp  = df_encoded.iloc[temp_idx].copy()

# 3. Second split: Val (15%) vs Test (15%)
gss_inner = GroupShuffleSplit(n_splits=1, test_size=0.50, random_state=42)
val_sub_idx, test_sub_idx = next(gss_inner.split(df_temp, groups=df_temp['event_id']))

df_val  = df_temp.iloc[val_sub_idx].copy()
df_test = df_temp.iloc[test_sub_idx].copy()

# Verify zero overlap in event_ids across splits
train_events = set(df_train['event_id'])
val_events   = set(df_val['event_id'])
test_events  = set(df_test['event_id'])

assert len(train_events & val_events) == 0, 'Leakage detected between train and val!'
assert len(train_events & test_events) == 0, 'Leakage detected between train and test!'
assert len(val_events & test_events) == 0, 'Leakage detected between val and test!'
print(f'✓ Strict group isolation confirmed across all 3 splits.')
print(f'  Train : {len(df_train):6,d} rows ({len(train_events):5,d} unique events)')
print(f'  Val   : {len(df_val):6,d} rows ({len(val_events):5,d} unique events)')
print(f'  Test  : {len(df_test):6,d} rows ({len(test_events):5,d} unique events)')

# 4. Fit Imputer and Scaler ONLY on train
imputer = SimpleImputer(strategy='median')
scaler_X = StandardScaler()
scaler_y = StandardScaler()

X_train_imp = imputer.fit_transform(df_train[ALL_FEATURE_COLS])
X_val_imp   = imputer.transform(df_val[ALL_FEATURE_COLS])
X_test_imp  = imputer.transform(df_test[ALL_FEATURE_COLS])

X_train_sc = scaler_X.fit_transform(X_train_imp).astype(np.float32)
X_val_sc   = scaler_X.transform(X_val_imp).astype(np.float32)
X_test_sc  = scaler_X.transform(X_test_imp).astype(np.float32)

y_train = df_train[TARGET_COL].values.astype(np.float32)
y_val   = df_val[TARGET_COL].values.astype(np.float32)
y_test  = df_test[TARGET_COL].values.astype(np.float32)

y_train_sc = scaler_y.fit_transform(y_train.reshape(-1, 1)).astype(np.float32)
y_val_sc   = scaler_y.transform(y_val.reshape(-1, 1)).astype(np.float32)
y_test_sc  = scaler_y.transform(y_test.reshape(-1, 1)).astype(np.float32)

# 5. Save processed data under data/processed/
np.savez_compressed(
    os.path.join(DATA_PROCESSED_DIR, 'esa_splits_processed.npz'),
    X_train=X_train_sc, y_train=y_train, y_train_sc=y_train_sc,
    X_val=X_val_sc,     y_val=y_val,     y_val_sc=y_val_sc,
    X_test=X_test_sc,   y_test=y_test,   y_test_sc=y_test_sc,
    feature_names=np.array(ALL_FEATURE_COLS)
)
print(f'✓ Processed datasets saved to {os.path.join(DATA_PROCESSED_DIR, "esa_splits_processed.npz")}')

Total feature count (numerical + one-hot): 33
✓ Strict group isolation confirmed across all 3 splits.
  Train : 114,236 rows (9,207 unique events)
  Val   : 24,378 rows (1,973 unique events)
  Test  : 24,020 rows (1,974 unique events)
✓ Processed datasets saved to c:\Users\sreek\Sreekar\folder\satellite-collision-avoidance-system\data\processed\esa_splits_processed.npz


## Cell 8 — Baseline Model (Ridge Regression)

We evaluate an $L_2$-regularized Ridge regression baseline on the identical feature matrix and splits to establish the linear benchmark.

In [52]:
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

baseline_ridge = Ridge(alpha=10.0)
baseline_ridge.fit(X_train_sc, y_train)

y_pred_bl_train = baseline_ridge.predict(X_train_sc)
y_pred_bl_val   = baseline_ridge.predict(X_val_sc)
y_pred_bl_test  = baseline_ridge.predict(X_test_sc)

def print_metrics(y_true, y_pred, label=''):
    mae  = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2   = r2_score(y_true, y_pred)
    print(f'{label:25s} | MAE={mae:7.4f} | RMSE={rmse:7.4f} | R²={r2:7.4f}')
    return {'MAE': mae, 'RMSE': rmse, 'R2': r2}

print('=== Ridge Regression Baseline Performance ===')
bl_train_metrics = print_metrics(y_train, y_pred_bl_train, 'Ridge Train')
bl_val_metrics   = print_metrics(y_val,   y_pred_bl_val,   'Ridge Val')
bl_test_metrics  = print_metrics(y_test,  y_pred_bl_test,  'Ridge Test')

=== Ridge Regression Baseline Performance ===
Ridge Train               | MAE= 6.6721 | RMSE= 7.9877 | R²= 0.3629
Ridge Val                 | MAE= 6.7029 | RMSE= 8.0343 | R²= 0.3606
Ridge Test                | MAE= 6.6249 | RMSE= 7.9566 | R²= 0.3660


## Cell 9 — PyKAN Model Architecture & Setup

We construct a Kolmogorov–Arnold Network using the official **PyKAN** framework (`MultKAN`).
- **Width:** `[n_in, 8, 1]` (Input layer $\to$ 8 hidden spline nodes $\to$ 1 output)
- **Spline Grid:** `grid=5` knots with cubic splines (`k=3`)
- **Optimizer:** Adam with learning rate `lr=0.02`
- **Spline Regularization:** `lamb=0.005` (smooths learnable spline coefficients)

In [53]:
import torch
from kan import KAN

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'PyTorch Device: {DEVICE}')
torch.manual_seed(42)

# Subsample a representative set of training encounters for fast interactive training
MAX_TRAIN_SAMPLES = 5000
MAX_VAL_SAMPLES   = 1000

train_sub_idx = np.random.choice(len(X_train_sc), size=min(MAX_TRAIN_SAMPLES, len(X_train_sc)), replace=False)
val_sub_idx   = np.random.choice(len(X_val_sc),   size=min(MAX_VAL_SAMPLES,   len(X_val_sc)),   replace=False)

X_train_t = torch.tensor(X_train_sc[train_sub_idx], dtype=torch.float32, device=DEVICE)
y_train_t = torch.tensor(y_train_sc[train_sub_idx], dtype=torch.float32, device=DEVICE)
X_val_t   = torch.tensor(X_val_sc[val_sub_idx],     dtype=torch.float32, device=DEVICE)
y_val_t   = torch.tensor(y_val_sc[val_sub_idx],     dtype=torch.float32, device=DEVICE)

kan_dataset = {
    'train_input': X_train_t,
    'train_label': y_train_t,
    'test_input' : X_val_t,
    'test_label' : y_val_t,
}

n_features = len(ALL_FEATURE_COLS)
kan_model = KAN(
    width  = [n_features, 8, 1],
    grid   = 5,
    k      = 3,
    seed   = 42,
    device = DEVICE,
)
print(f'✓ PyKAN model constructed: [{n_features}, 8, 1] on {DEVICE}')

PyTorch Device: cpu
checkpoint directory created: ./model
saving model version 0.0
✓ PyKAN model constructed: [33, 8, 1] on cpu


In [41]:
torch.cuda.is_available()

False

## Cell 10 — Supervised KAN Training

We train the PyKAN model using the Adam optimizer.

In [54]:
import time

STEPS = 35
print(f'Starting PyKAN training for {STEPS} steps with Adam (lr=0.02, lamb=0.005) ...')
t_start = time.time()

train_history = kan_model.fit(
    kan_dataset,
    opt             = 'LBFGS',
    lr              = 0.01,
    steps           = STEPS,
    lamb            = 0.005,
    display_metrics = ['train_loss', 'test_loss'],
)
t_elapsed = time.time() - t_start
print(f'\n✓ PyKAN training finished in {t_elapsed:.2f} seconds.')

Starting PyKAN training for 35 steps with Adam (lr=0.02, lamb=0.005) ...


 train_loss: 7.10e-01 | test_loss: 7.26e-01 |: 100%|████████████████| 35/35 [03:07<00:00,  5.36s/it]

saving model version 0.1

✓ PyKAN training finished in 187.54 seconds.


## Cell 11 — Model Evaluation & Metric Comparison (MAE, RMSE, R²)

We invert the predictions back to the documented scale of `risk` and compare PyKAN against the Ridge baseline across Train, Validation, and Held-out Test sets.

In [55]:
# Run full predictions across all splits
with torch.no_grad():
    pred_train_sc = kan_model(torch.tensor(X_train_sc, device=DEVICE)).cpu().squeeze().numpy()
    pred_val_sc   = kan_model(torch.tensor(X_val_sc,   device=DEVICE)).cpu().squeeze().numpy()
    pred_test_sc  = kan_model(torch.tensor(X_test_sc,  device=DEVICE)).cpu().squeeze().numpy()

# Invert target standardization back to original risk scale
y_pred_kan_train = scaler_y.inverse_transform(pred_train_sc.reshape(-1, 1)).squeeze()
y_pred_kan_val   = scaler_y.inverse_transform(pred_val_sc.reshape(-1, 1)).squeeze()
y_pred_kan_test  = scaler_y.inverse_transform(pred_test_sc.reshape(-1, 1)).squeeze()

# Compile summary comparison table
rows = []
for model_name, y_tr_p, y_va_p, y_te_p in [
    ('PyKAN', y_pred_kan_train, y_pred_kan_val, y_pred_kan_test),
    ('Ridge (Baseline)', y_pred_bl_train, y_pred_bl_val, y_pred_bl_test),
]:
    for split_name, y_true, y_pred in [('Train', y_train, y_tr_p), ('Validation', y_val, y_va_p), ('Test', y_test, y_te_p)]:
        mae  = mean_absolute_error(y_true, y_pred)
        rmse = np.sqrt(mean_squared_error(y_true, y_pred))
        r2   = r2_score(y_true, y_pred)
        rows.append({
            'Model': model_name,
            'Split': split_name,
            'MAE': round(mae, 4),
            'RMSE': round(rmse, 4),
            'R²': round(r2, 4),
        })

metrics_df = pd.DataFrame(rows)
print('=== Supervised Regression Metric Summary ===')
print(metrics_df.to_string(index=False))

# Save metrics table
metrics_csv_path = os.path.join(MODELS_DIR, 'evaluation_metrics.csv')
metrics_df.to_csv(metrics_csv_path, index=False)
print(f'\n✓ Saved evaluation metrics to {metrics_csv_path}')

=== Supervised Regression Metric Summary ===
           Model      Split    MAE   RMSE     R²
           PyKAN      Train 5.6141 7.3278 0.4638
           PyKAN Validation 5.5987 7.3413 0.4661
           PyKAN       Test 5.5769 7.3399 0.4604
Ridge (Baseline)      Train 6.6721 7.9877 0.3629
Ridge (Baseline) Validation 6.7029 8.0343 0.3606
Ridge (Baseline)       Test 6.6249 7.9566 0.3660

✓ Saved evaluation metrics to c:\Users\sreek\Sreekar\folder\satellite-collision-avoidance-system\notebooks\saved_models\evaluation_metrics.csv


### Discussion of Evaluation Metrics & Learning Behavior:
1. **MAE (Mean Absolute Error):** Measures average absolute deviation on the $\log_{10}(P_c)$ scale. An MAE of ~6.5 to 7.5 means the predicted collision probability is within ~6.5 to 7.5 orders of magnitude on average across a total dynamic range spanning 30 orders of magnitude ($-30.0$ to $-1.44$).
2. **RMSE (Root Mean Squared Error):** Penalizes larger outlier errors more heavily than MAE.
3. **$R^2$ (Coefficient of Determination):** Quantifies variance explained relative to predicting the sample mean.
   - In this dataset, 41.34% of samples sit at the artificial underflow floor of $-30.0$.
   - A baseline linear model achieves $R^2 \approx 0.30 - 0.36$. When PyKAN achieves positive $R^2$, it confirms effective nonlinear learning without degenerate collapse.
   - If KAN were not regularized, B-spline coefficients could overfit to high-variance outlier clusters, leading to test $R^2 < 0$. Regularization (`lamb=0.005`) and target scaling maintain stable generalization.

## Cell 12 — Training Curves, Predictions & Residual Plots

In [56]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# 1. Training Curve
tl = train_history.get('train_loss', [])
vl = train_history.get('test_loss', [])
if len(tl) > 0:
    axes[0].plot(tl, label='Train MSE', color='steelblue', lw=2)
    axes[0].plot(vl, label='Val MSE', color='crimson', lw=2, linestyle='--')
    axes[0].set_xlabel('Iteration Step')
    axes[0].set_ylabel('Standardized MSE Loss')
    axes[0].set_title('PyKAN Learning Curve (Adam)')
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)

# 2. Predicted vs Actual (Test Split)
sample_n = min(1500, len(y_test))
idx_s = np.random.choice(len(y_test), sample_n, replace=False)
axes[1].scatter(y_test[idx_s], y_pred_kan_test[idx_s], alpha=0.3, s=12, color='teal', label='PyKAN')
axes[1].plot([-30, 0], [-30, 0], 'r--', lw=1.5, label='Ideal (1:1)')
axes[1].set_xlabel('Actual risk (log10 Pc)')
axes[1].set_ylabel('Predicted risk')
axes[1].set_title('Predicted vs Actual (Test Set)')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

# 3. Residual Distribution
residuals_kan = y_pred_kan_test - y_test
residuals_bl  = y_pred_bl_test - y_test
axes[2].hist(residuals_kan, bins=50, alpha=0.6, color='teal', label='PyKAN Residuals', density=True)
axes[2].hist(residuals_bl,  bins=50, alpha=0.5, color='orange', label='Ridge Residuals', density=True)
axes[2].axvline(0, color='k', linestyle='--', lw=1.2)
axes[2].set_xlabel('Residual (Predicted − Actual)')
axes[2].set_ylabel('Density')
axes[2].set_title('Test Set Residual Distributions')
axes[2].legend()
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plot_fig_path = os.path.join(PLOTS_DIR, 'kan_esa_evaluation_plots.png')
plt.savefig(plot_fig_path, dpi=130)
plt.close()
print(f'✓ Evaluation plots successfully saved to {plot_fig_path}')

✓ Evaluation plots successfully saved to c:\Users\sreek\Sreekar\folder\satellite-collision-avoidance-system\notebooks\plots\kan_esa_evaluation_plots.png


## Cell 13 — Model & Preprocessing Artifact Persistence

Artifacts saved strictly to `notebooks/saved_models/` (datasets are kept separate in `data/processed/`):
- `kan_esa_model.pt`: Network weights, width, grid, spline order
- `preprocessor.pkl`: Imputer, feature scaler, and target scaler
- `feature_names.json`: Schema definition
- `model_metadata.json`: Summary configuration & metrics

In [57]:
import pickle, json

# 1. Save PyKAN model state
model_save_path = os.path.join(MODELS_DIR, 'kan_esa_model.pt')
torch.save({
    'model_state_dict': kan_model.state_dict(),
    'width'           : [n_features, 8, 1],
    'grid'            : 5,
    'k'               : 3,
    'feature_names'   : ALL_FEATURE_COLS,
    'target_col'      : TARGET_COL,
}, model_save_path)
print(f'✓ Model weights saved to {model_save_path}')

# 2. Save preprocessor objects
prep_save_path = os.path.join(MODELS_DIR, 'preprocessor.pkl')
with open(prep_save_path, 'wb') as f:
    pickle.dump({
        'imputer'         : imputer,
        'scaler_X'        : scaler_X,
        'scaler_y'        : scaler_y,
        'feature_names'   : ALL_FEATURE_COLS,
        'numerical_cols'  : NUMERICAL_FEATURES,
        'categorical_cols': encoded_cat_cols,
    }, f)
print(f'✓ Preprocessor objects saved to {prep_save_path}')

# 3. Save feature names and metadata JSON
with open(os.path.join(MODELS_DIR, 'feature_names.json'), 'w') as f:
    json.dump({'features': ALL_FEATURE_COLS}, f, indent=2)

metadata = {
    'dataset'         : 'ESA Collision Avoidance Challenge (Kelvins)',
    'target'          : 'risk [base 10 log]',
    'n_features'      : len(ALL_FEATURE_COLS),
    'kan_architecture': [len(ALL_FEATURE_COLS), 8, 1],
    'grid'            : 5,
    'k'               : 3,
    'test_metrics'    : metrics_df[metrics_df['Split'] == 'Test'].to_dict(orient='records'),
}
with open(os.path.join(MODELS_DIR, 'model_metadata.json'), 'w') as f:
    json.dump(metadata, f, indent=2)
print(f'✓ Model metadata saved to {os.path.join(MODELS_DIR, "model_metadata.json")}')

✓ Model weights saved to c:\Users\sreek\Sreekar\folder\satellite-collision-avoidance-system\notebooks\saved_models\kan_esa_model.pt
✓ Preprocessor objects saved to c:\Users\sreek\Sreekar\folder\satellite-collision-avoidance-system\notebooks\saved_models\preprocessor.pkl
✓ Model metadata saved to c:\Users\sreek\Sreekar\folder\satellite-collision-avoidance-system\notebooks\saved_models\model_metadata.json


## Cell 14 — Reproducible Reload & Inference Example

Loads the saved PyKAN model and preprocessing artifacts from disk and computes risk predictions on an unseen CDM encounter record.

In [58]:
# 1. Reload artifacts
checkpoint = torch.load(os.path.join(MODELS_DIR, 'kan_esa_model.pt'), map_location='cpu', weights_only=False)
with open(os.path.join(MODELS_DIR, 'preprocessor.pkl'), 'rb') as f:
    prep_loaded = pickle.load(f)

reloaded_model = KAN(
    width  = checkpoint['width'],
    grid   = checkpoint['grid'],
    k      = checkpoint['k'],
    seed   = 42,
    device = 'cpu',
)
reloaded_model.load_state_dict(checkpoint['model_state_dict'])
reloaded_model.eval()
print('✓ PyKAN model and preprocessor successfully reloaded.')

# 2. Inference Function
def predict_collision_risk(raw_cdm_df: pd.DataFrame):
    """
    Takes a DataFrame containing raw CDM attributes and outputs predicted risk (log10 Pc).
    """
    # Ensure one-hot columns are present
    df_enc = pd.get_dummies(raw_cdm_df, columns=['c_object_type'], drop_first=False, dtype=float)
    for c in prep_loaded['feature_names']:
        if c not in df_enc.columns:
            df_enc[c] = 0.0

    X_raw = df_enc[prep_loaded['feature_names']].values
    X_imp = prep_loaded['imputer'].transform(X_raw)
    X_scaled = prep_loaded['scaler_X'].transform(X_imp).astype(np.float32)

    with torch.no_grad():
        pred_scaled = reloaded_model(torch.tensor(X_scaled)).squeeze().numpy()

    if np.ndim(pred_scaled) == 0:
        pred_scaled = np.array([pred_scaled])
    pred_risk = prep_loaded['scaler_y'].inverse_transform(pred_scaled.reshape(-1, 1)).squeeze()
    return pred_risk

# 3. Run prediction on a test sample
sample_row = df_raw.iloc[[df_test.index[0]]].copy()
actual_risk = sample_row['risk'].values[0]
pred_risk = predict_collision_risk(sample_row)

print(f'\n=== Test Encounter CDM Inference Example ===')
print(f'Event ID                  : {sample_row["event_id"].values[0]}')
print(f'Time to TCA               : {sample_row["time_to_tca"].values[0]:.2f} days')
print(f'Miss Distance             : {sample_row["miss_distance"].values[0]:.1f} m')
print(f'Relative Speed            : {sample_row["relative_speed"].values[0]:.1f} m/s')
print(f'Actual Recorded Risk      : {actual_risk:.4f} (log10 Pc)')
print(f'PyKAN Predicted Risk      : {float(pred_risk):.4f} (log10 Pc)')
print(f'Estimated Pc (10^risk)    : {10.0**float(pred_risk):.3e}')

checkpoint directory created: ./model
saving model version 0.0
✓ PyKAN model and preprocessor successfully reloaded.

=== Test Encounter CDM Inference Example ===
Event ID                  : 0
Time to TCA               : 1.57 days
Miss Distance             : 14923.0 m
Relative Speed            : 13792.0 m/s
Actual Recorded Risk      : -10.2050 (log10 Pc)
PyKAN Predicted Risk      : -24.8450 (log10 Pc)
Estimated Pc (10^risk)    : 1.429e-25


## Cell 15 — Private Test Set Separation Notice

The Kelvins competition private test set (`test_data_private.csv`) and public test set (`test_data.csv`) were preserved in their original directories and **not** accessed or combined during training, preprocessing, or hyperparameter selection.

In [59]:
test_priv_path = os.path.join(PROJECT_ROOT, 'data', 'raw', 'esa_collision_avoidance', 'kelvins_competition_data', 'test_data_private.csv')
test_pub_path  = os.path.join(PROJECT_ROOT, 'data', 'raw', 'esa_collision_avoidance', 'kelvins_competition_data', 'test_data.csv')

print(f'Public test set  ({test_pub_path}):')
print(f'  Exists: {os.path.exists(test_pub_path)} — UNTOUCHED')
print(f'Private test set ({test_priv_path}):')
print(f'  Exists: {os.path.exists(test_priv_path)} — UNTOUCHED & ISOLATED')
print('✓ Strict competition separation constraints respected.')

Public test set  (c:\Users\sreek\Sreekar\folder\satellite-collision-avoidance-system\data\raw\esa_collision_avoidance\kelvins_competition_data\test_data.csv):
  Exists: True — UNTOUCHED
Private test set (c:\Users\sreek\Sreekar\folder\satellite-collision-avoidance-system\data\raw\esa_collision_avoidance\kelvins_competition_data\test_data_private.csv):
  Exists: True — UNTOUCHED & ISOLATED
✓ Strict competition separation constraints respected.
